# **🏠 Regresión lineal simple y múltiple**

**Caso de estudio:** California Housing · **Herramientas:** NumPy, pandas y scikit-learn

En este notebook vamos a trabajar con el dataset **California Housing**, disponible en `scikit-learn`.  
El objetivo es **aplicar regresión lineal simple y múltiple** para predecir el valor medio de las viviendas en distintas áreas de California.

### **🧭 Guía de contenidos**


1. **Importación de librerías**  
   Se cargan todas las librerías necesarias para análisis, modelado, evaluación y visualización.

2. **Entendimiento con ejemplo sencillo**
   Se considera un ejemplo con $5$ puntos, se deduce como llegar a la solución y se modela con ayuda de `numpy`.

2. **Carga del dataset**  
   Usamos `fetch_california_housing(as_frame=True)`, que nos devuelve un objeto tipo *Bunch* con los datos en un DataFrame (`.frame`) y la variable objetivo (`.target`).

3. **Exploración de datos (EDA breve)**  
   - Dimensiones del dataset.  
   - Primeras filas.  
   - Estadísticas descriptivas.  
   - Correlación de las variables predictoras con la variable objetivo.  

4. **Regresión Lineal Simple**  
   - Seleccionamos solo una variable predictora (`MedInc`, ingreso medio).  
   - Entrenamos el modelo y evaluamos con métricas (`MSE`, `R²`).  
   - Visualizamos la recta de regresión ajustada sobre los datos de prueba.  

5. **Regresión Lineal Múltiple**  
   - Usamos **todas las variables predictoras** disponibles en el dataset.  
   - Entrenamos el modelo, obtenemos métricas y coeficientes.  
   - Graficamos valores reales vs valores predichos para evaluar el ajuste.

### **📐 1. Fundamentos: regresión lineal con NumPy**


La regresión lineal es una técnica estadística y de análisis de datos que modela la relación entre una variable dependiente $Y$ y una o más variables independientes $X$ mediante una ecuación en forma lineal en sus parámetros.

$$y_i=\beta_0+\beta_1x_i+\varepsilon_i, \qquad \widehat y_i=\beta_0+\beta_1x_i.$$

- $\beta_0$: intercepto, valor predicho cuando $x=0$.
- $\beta_1$: pendiente, cambio en la predicción por cada unidad adicional de $x$.
- $\varepsilon_i$: error del modelo; una vez ajustado, el residuo observado es $e_i=y_i-\widehat y_i$.

Usaremos los puntos: $(1, 2), (2, 4), (3, 5), (4, 4), (5, 5)$

In [ ]:
import numpy as np

x_5 = np.array([1, 2, 3, 4, 5], dtype=float)
y_5 = np.array([2, 4, 5, 4, 5], dtype=float).reshape(-1, 1)
X_5 = np.column_stack((np.ones(len(x_5)), x_5))

print("X =\n", X_5)
print("y =\n", y_5)
print("Dimensiones de X e y:", X_5.shape, y_5.shape)

##### **🎯 1.1. Función de costo**


La función de costo mide qué tan lejos están las predicciones de los valores observados. Usaremos el **error cuadrático medio (MSE)**:

$$J(\boldsymbol\beta)=\frac{1}{n}\sum_{i=1}^{n}(y_i-\widehat y_i)^2
=\frac{1}{n}(\mathbf y-\mathbf X\boldsymbol\beta)^T(\mathbf y-\mathbf X\boldsymbol\beta).$$

Los cuadrados evitan que los residuos positivos y negativos se cancelen y dan mayor peso a errores grandes. Ajustar el modelo significa encontrar los betas que minimizan este costo. Minimizar la suma de cuadrados (SSE) produce los mismos betas, porque el factor $1/n$ es constante.

##### **🧮 1.2. Representación matricial y solución**


$$\mathbf y=\mathbf X\boldsymbol\beta+\boldsymbol\varepsilon,\qquad
\mathbf X=\begin{pmatrix}1&1\\1&2\\1&3\\1&4\\1&5\end{pmatrix},\quad
\mathbf y=\begin{pmatrix}2\\4\\5\\4\\5\end{pmatrix},\quad
\boldsymbol\beta=\begin{pmatrix}\beta_0\\\beta_1\end{pmatrix}.$$

Cada fila de $\mathbf X$ representa un punto. La primera columna contiene unos para incorporar el intercepto y la segunda contiene los valores de $x$. Las dimensiones son $\mathbf X$: $(5,2)$, $\mathbf y$: $(5,1)$ y $\boldsymbol\beta$: $(2,1)$.

La solución de mínimos cuadrados es:

$$\boxed{\widehat{\boldsymbol\beta}=(\mathbf X^T\mathbf X)^{-1}\mathbf X^T\mathbf y}.$$

$T$ indica transposición y $^{-1}$ indica la matriz inversa. La fórmula requiere que las columnas de $\mathbf X$ sean linealmente independientes. En este ejemplo se cumple porque $x$ toma valores distintos. En NumPy, `@` representa el producto matricial.

##### **🔢 1.3. Cálculo paso a paso**


Calculamos primero $\mathbf X^T\mathbf X$ y $\mathbf X^T\mathbf y$, y luego aplicamos la fórmula.

In [ ]:
XtX_5 = X_5.T @ X_5
Xty_5 = X_5.T @ y_5
XtX_inv_5 = np.linalg.inv(XtX_5)
beta_5 = XtX_inv_5 @ Xty_5

print("X.T @ X =\n", XtX_5)
print("X.T @ y =\n", Xty_5)
print("Inversa de X.T @ X =\n", XtX_inv_5)
print("Betas [intercepto, pendiente] =\n", beta_5)
print(f"Recta ajustada: y_hat = {beta_5[0, 0]:.1f} + {beta_5[1, 0]:.1f} * x")

# Alternativa que evita construir explícitamente la inversa:
beta_solve_5 = np.linalg.solve(XtX_5, Xty_5)
print("Betas usando solve =\n", beta_solve_5)

El resultado es $\widehat\beta_0=2.2$ y $\widehat\beta_1=0.6$, de modo que **$\widehat y=2.2+0.6x$**. Por cada unidad adicional de $x$, la predicción aumenta 0.6 unidades. El intercepto 2.2 corresponde a $x=0$, que está fuera de los cinco valores observados.

Mostramos la inversa para reproducir la fórmula. Para cálculos numéricos, `np.linalg.solve` evita construirla; en problemas generales, `np.linalg.lstsq(X, y, rcond=None)` permite resolver mínimos cuadrados directamente sin formar $\mathbf X^T\mathbf X$.

##### **📊 1.4. Predicciones, residuos y costo**



Multiplicamos $\mathbf X\widehat{\boldsymbol\beta}$ y evaluamos la función de costo sobre estos cinco puntos.

In [ ]:
y_hat_5 = X_5 @ beta_5
residuos_5 = y_5 - y_hat_5
sse_5 = np.sum(residuos_5 ** 2)
mse_5 = np.mean(residuos_5 ** 2)

print("    x      y   y_hat  residuo")
print(np.round(np.column_stack((x_5, y_5, y_hat_5, residuos_5)), 2))
print(f"SSE = {sse_5:.2f}")
print(f"Costo J(beta) = MSE = {mse_5:.2f}")

Las predicciones son **2.8, 3.4, 4.0, 4.6 y 5.2**. La suma de cuadrados es **SSE = 2.4** y el costo es **MSE = 2.4 / 5 = 0.48**. Estos betas minimizan el costo entre todas las rectas posibles para los cinco puntos. Este es un costo de ajuste sobre los mismos datos usados para estimar los betas; la evaluación con datos de prueba aparece más adelante en el notebook.

### **🗂️ 2. Preparación de datos**


##### **📦 2.1. Importación de librerías**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

##### **📥 2.2. Carga del conjunto de datos**


`fetch_california_housing(as_frame=True)` devuelve un objeto tipo **Bunch**, que contiene:

* **`frame`**: un `DataFrame` con las variables predictoras (X).  
* **`target`**: una serie/columna objetivo (y), llamada `MedHouseVal`.  

**Nota:** el parámetro `as_frame=True` indica que los datos deben entregarse directamente como un `pandas.DataFrame`.  
Si no se incluye, el método devuelve arreglos de `NumPy`, lo que puede ser menos práctico para la exploración y manipulación inicial de los datos.  

In [ ]:
df = data.frame.copy()

# Nota: usamos .copy() para crear un DataFrame independiente del original (data.frame).
# Si hacemos df = data.frame, ambos apuntan al mismo objeto en memoria.
# Entonces, cualquier cambio en df (ej. añadir o eliminar columnas) también afectaría a data.frame.
# Con .copy() evitamos este "aliasing" y trabajamos con una copia segura.

df["MedHouseVal"] = data.target  # añadimos la variable objetivo explícitamente
df.head()

In [ ]:
# Nombres de columnas (features) y objetivo
feature_names = data.feature_names
target_name = "MedHouseVal"

##### **🔎 2.3. Análisis exploratorio de datos (EDA)**


In [ ]:
print(f"Dimensiones del DataFrame: {df.shape}")
print("\nPrimeras filas:")
df.head()

In [ ]:
print("\nDescripción estadística de features (X):")
df[feature_names].describe()

In [ ]:
print("\nDescripción estadística de la variable objetivo (y):")
df[[target_name]].describe()

In [ ]:
# Revisión rápida de valores faltantes
print("\nConteo de valores faltantes por columna:")
df.isna().sum()

In [ ]:
# Correlación de features con el objetivo
print("\nCorrelación (Pearson) de cada feature con el objetivo:")
corr_with_target = df[feature_names].corrwith(df[target_name]).sort_values(ascending=False)
corr_with_target

##### **📏 2.4. Ajuste matricial con el ingreso medio**

Aplicamos la fórmula de mínimos cuadrados usando `MedInc` como variable explicativa:

$$\widehat{\boldsymbol\beta} = (X^T X)^{-1}X^T y$$

In [ ]:
X = np.array([[1, mi] for mi in df['MedInc']])
y = np.array(df['MedHouseVal'])

beta = np.linalg.inv(X.T @ X) @ X.T @ y
beta

In [ ]:
sns.scatterplot(data=df, x="MedInc", y="MedHouseVal", alpha=0.5)
x_linea = np.linspace(df["MedInc"].min(), df["MedInc"].max(), 100)
y_linea = beta[0] + beta[1] * x_linea  # Intercepto + pendiente * x
plt.plot(x_linea, y_linea, color="red", label="Recta ajustada")
plt.xlabel("Ingreso medio (MedInc)")
plt.ylabel("Valor medio de vivienda (MedHouseVal)")
plt.legend()
plt.show()

### **📈 3. Regresión lineal simple con scikit-learn**


Usamos una sola variable predictora, **`MedInc`** (ingreso medio), para estimar **`MedHouseVal`** (valor medio de vivienda).

In [ ]:
# Elección de la feature: MedInc
X_simple = df[["MedInc"]]

# Asignación de la variable objetivo
y = df[target_name]

In [ ]:
# Dividimos en entrenamiento y prueba para evaluar generalización
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple, y, test_size=0.2, random_state=42
)

In [ ]:
# Creamos y entrenamos el modelo de regresión lineal
lr_simple = LinearRegression()
lr_simple.fit(X_train_s, y_train_s)  # Entrenamiento

In [ ]:
# Predicciones sobre el conjunto de prueba
y_pred_s = lr_simple.predict(X_test_s)


##### **📉 3.1. Evaluación y visualización**


La evaluación se realiza con los datos de **prueba**, que no participaron en el entrenamiento. Así comprobamos qué tan bien generaliza el modelo a observaciones nuevas.

**Error cuadrático medio (MSE).** Mide el promedio de los errores de predicción elevados al cuadrado:

$$
\mathrm{MSE}=\frac{1}{n}\sum_{i=1}^{n}(y_i-\widehat y_i)^2.
$$

Aquí, $n$ es el número de observaciones de prueba, $y_i$ el valor real y $\widehat y_i$ la predicción. Al elevar los errores al cuadrado, los fallos grandes reciben más peso. **Cuanto menor es el MSE, mejor**; un valor de cero indica predicciones exactas. Sus unidades son las de la variable objetivo al cuadrado.


**Coeficiente de determinación ($R^2$).** Compara el error del modelo con el de predecir siempre la media de los valores observados:

$$
R^2=1-\frac{\mathrm{SS}_{\mathrm{res}}}{\mathrm{SS}_{\mathrm{tot}}},
\qquad
\mathrm{SS}_{\mathrm{res}}=\sum_{i=1}^{n}(y_i-\widehat y_i)^2,
\qquad
\mathrm{SS}_{\mathrm{tot}}=\sum_{i=1}^{n}(y_i-\bar y)^2.
$$

$\mathrm{SS}_{\mathrm{res}}$ representa la variación que las predicciones no logran explicar, $\mathrm{SS}_{\mathrm{tot}}$ la variación total respecto de la media $\bar y$. Un **$R^2=1$** indica ajuste perfecto, **$R^2=0$** equivale al rendimiento de predecir la media del conjunto evaluado y **$R^2<0$** significa que el modelo rinde peor que esa referencia. Cuanto más alto, mejor, aunque no basta por sí solo para afirmar que un modelo es adecuado.

**Visualización.** En el diagrama se representan los valores reales frente al ingreso medio (`MedInc`) y se superpone la recta estimada. La proximidad de los puntos a la recta indica menores residuos $e_i=y_i-\widehat y_i$; una dispersión amplia indica mayor error. El gráfico permite detectar patrones y desviaciones que una única métrica podría ocultar.

In [ ]:
# Métricas de evaluación en test
mse_s = mean_squared_error(y_test_s, y_pred_s)  # Error cuadrático medio
r2_s = r2_score(y_test_s, y_pred_s)  # Coeficiente de terminación

In [ ]:
print(f"Intercepto: {lr_simple.intercept_}")
print(f"Coeficiente (MedInc): {lr_simple.coef_[0]}")
print(f"MSE (test): {mse_s:.4f}")
print(f"R^2 (test): {r2_s:.4f}")

In [ ]:
# Ordenar los datos por ingreso para trazar la recta sin saltos
order_idx = np.argsort(X_test_s.values.squeeze())
X_plot = X_test_s.values.squeeze()[order_idx]
y_plot = y_test_s.values.squeeze()[order_idx]
y_pred_plot = y_pred_s.squeeze()[order_idx]

In [ ]:
plt.figure()
plt.scatter(X_test_s.values.squeeze(), y_test_s, alpha=0.6, label="Datos (test)")
plt.plot(X_plot, y_pred_plot, label="Recta ajustada", color="red")
plt.title("Regresión lineal simple: MedInc vs MedHouseVal (Test)")
plt.xlabel("Ingreso medio ×1000")
plt.ylabel("Valor medio de vivienda (×100000)")
plt.legend()
plt.show()

### **🧩 4. Regresión lineal múltiple**

En este modelo utilizamos **todas las variables predictoras** del conjunto de datos para estimar `MedHouseVal`.

In [ ]:
X = df[feature_names]

# Dividimos en entrenamiento y prueba
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Creamos y entrenamos el modelo de regresión lineal múltiple
lr_multi = LinearRegression()
lr_multi.fit(X_train_m, y_train_m)

In [ ]:
# Predicciones en test
y_pred_m = lr_multi.predict(X_test_m)

##### **📉 4.1. Evaluación y visualización**


La evaluación usa el **mismo conjunto de prueba** que la regresión simple, lo que permite comparar ambos modelos.

**MSE (error cuadrático medio):**

$$
\mathrm{MSE}=\frac{1}{n}\sum_{i=1}^{n}(y_i-\widehat y_i)^2.
$$

Es el promedio del cuadrado de la diferencia entre el valor real $y_i$ y la predicción $\widehat y_i$. $n$ es el número de casos evaluados. **Valores menores indican errores más pequeños** y los errores grandes tienen mayor penalización.

**$R^2$ (coeficiente de determinación):**

$$
R^2=1-\frac{\sum_{i=1}^{n}(y_i-\widehat y_i)^2}{\sum_{i=1}^{n}(y_i-\bar y)^2}.
$$

La cantidad $\bar y$ es la media de los valores reales del conjunto de prueba. El numerador suma los errores cuadrados del modelo y el denominador expresa la variación respecto de esa media. **$R^2=1$** es perfecto, **$R^2=0$** iguala la predicción constante por la media y **$R^2<0$** indica un desempeño inferior a esa referencia.

**Gráfico de valores reales frente a predichos.** Cada punto tiene coordenadas $(y_i,\widehat y_i)$. La diagonal $y=x$ representa las predicciones perfectas; **cuanto más cerca estén los puntos de ella, mejor será el ajuste**. La dispersión o los patrones sistemáticos alrededor de la diagonal pueden revelar errores importantes o sesgos. Además, los coeficientes estimados permiten examinar la dirección de la relación de cada variable con el objetivo, manteniendo las demás constantes.


In [ ]:
# Métricas de evaluación en test
mse_m = mean_squared_error(y_test_m, y_pred_m)
r2_m = r2_score(y_test_m, y_pred_m)

In [ ]:
print(f"Intercepto: {lr_multi.intercept_}")
print(f"MSE (test): {mse_m:.4f}")
print(f"R^2 (test): {r2_m:.4f}")

In [ ]:
# Coeficientes por variable (útil para interpretar dirección e importancia relativa)
coefs_df = pd.DataFrame({
    "feature": feature_names,
    "coef": lr_multi.coef_
})

coefs_df

In [ ]:
# Visualización: y real vs y predicho en test (línea y=x como referencia ideal)
plt.figure()
plt.scatter(y_test_m, y_pred_m, alpha=0.6, label="Predicción (test)")
min_y = min(y_test_m.min(), y_pred_m.min())
max_y = max(y_test_m.max(), y_pred_m.max())
plt.plot([min_y, max_y], [min_y, max_y], linestyle="--", label="Referencia: y = x")
plt.title("Regresión lineal múltiple: Valores reales vs predichos (Test)")
plt.xlabel("MedHouseVal real")
plt.ylabel("MedHouseVal predicho")
plt.legend()
plt.show()